In [ ]:
# ============================================================
# Prototype (Template)-Based Authentication Experiment
# ============================================================
#
# This script implements the prototype/template-based authentication system.
#
# Pipeline:
#   1. Load a trained classifier.
#   2. Extract embeddings from the layer before the classifier's
#      final Dense layer.
#   3. Build one prototype/template per user from enrollment data.
#   4. Verify claims using cosine similarity between a test
#      embedding and the claimed user's prototype.
#   5. Tune the global verification threshold on a development set.
#   6. Evaluate the final system on the held-out test set.
#   7. Study the effect of enrollment size (k).
#
# Data usage:
#   SET1 (X_set1, y_set1): enrollment / template creation
#   SET2 (X_set2, y_set2): split into development and final test
#
# Outputs:
#   CSV: trial scores, summaries, per-user metrics,
#        EER vs enrollment size
#   JSON: experiment configuration and threshold information
#   Pickle: prototypes and summaries
#   NPY: ROC curves and EER-vs-enrollment arrays
#   PNG/PDF: ROC and enrollment-size plots
# ============================================================

import os
import gc

# Make CUDA visible before TensorFlow is loaded.
os.environ["CUDA_VISIBLE_DEVICES"] = "1,3,6"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import json
import pickle
import datetime
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling1D

In [ ]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class ExperimentConfig:
    dev_test_split: float = 0.5
    random_state: int = 42
    enroll_sizes: Tuple[int, ...] = (1, 2, 3, 5, 10, 15)
    batch_size: int = 32
    use_pooling_for_embedding: bool = False


class ExperimentPaths:
    def __init__(self, exp_root: str, exp_name: str):
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_dir = os.path.join(exp_root, f"exp_{ts}_{exp_name}")
        self.config_dir = os.path.join(self.exp_dir, "config")
        self.data_dir = os.path.join(self.exp_dir, "data")
        self.plots_dir = os.path.join(self.exp_dir, "plots")

        for d in [
            self.exp_dir,
            self.config_dir,
            self.data_dir,
            self.plots_dir,
        ]:
            os.makedirs(d, exist_ok=True)

    def path_config(self, name: str) -> str:
        return os.path.join(self.config_dir, name)

    def path_data(self, name: str) -> str:
        return os.path.join(self.data_dir, name)

    def path_plot(self, name: str) -> str:
        return os.path.join(self.plots_dir, name)


# ============================================================
# Utility functions
# ============================================================

def l2_normalize(
    x: np.ndarray,
    axis: int = -1,
    eps: float = 1e-9, # avoids devision by 0
) -> np.ndarray:
    norm = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (norm + eps)


def compute_eer_threshold(
    labels: np.ndarray,
    scores: np.ndarray,
) -> Tuple[float, float, float, np.ndarray, np.ndarray, np.ndarray]:

    fpr, tpr, thresholds = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)

    fnr = 1.0 - tpr
    idx_eer = np.argmin(np.abs(fpr - fnr))

    eer_threshold = thresholds[idx_eer]
    eer = (fpr[idx_eer] + fnr[idx_eer]) / 2.0

    return (
        float(eer_threshold),
        float(eer),
        float(roc_auc),
        fpr,
        tpr,
        thresholds,
    )


def eval_at_threshold(
    labels: np.ndarray,
    scores: np.ndarray,
    threshold: float,
) -> Dict[str, float]:

    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)

    predictions = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    acc = accuracy_score(labels, predictions)

    return {
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "FAR": float(far),
        "FRR": float(frr),
        "ACC": float(acc),
    }

In [ ]:
# ============================================================
# Embedding extractor
# ============================================================

class EmbeddingExtractor:
    """
    Extracts the representation immediately before the final
    classification layer of the supplied trained model.

    The classifier itself is NOT used for authentication scores.
    It is used only as the source of the embedding representation.
    """

    def __init__(
        self,
        classifier_model: tf.keras.Model,
        use_pooling: bool = False,
    ):
        self.classifier_model = classifier_model
        self.use_pooling = use_pooling
        self.embedding_model = self._build_embedding_model()

    def _build_embedding_model(self) -> tf.keras.Model:
        final_layer = self.classifier_model.layers[-1]
        pre_classifier_tensor = final_layer.input

        if (
            self.use_pooling
            and len(pre_classifier_tensor.shape) == 3
        ):
            embedding_out = GlobalAveragePooling1D()(
                pre_classifier_tensor
            )
        else:
            embedding_out = pre_classifier_tensor

        return Model(
            inputs=self.classifier_model.inputs,
            outputs=embedding_out,
            name="embedding_model",
        )

    def embed(
        self,
        X: np.ndarray,
        batch_size: int = 32,
    ) -> np.ndarray:

        embeddings = self.embedding_model.predict(
            X,
            batch_size=batch_size,
            verbose=0,
        )

        return l2_normalize(embeddings, axis=1)


# ============================================================
# Prototype / template management
# ============================================================

class PrototypeManager:
    """
    Creates one normalized prototype/template for each user.

    Prototype = normalized mean of the user's normalized
    enrollment embeddings.
    """

    def __init__(
        self,
        embedding_extractor: EmbeddingExtractor,
        users: np.ndarray,
    ):
        self.embedding_extractor = embedding_extractor
        self.users = np.asarray(users, dtype=int)
        self.prototypes: Dict[int, np.ndarray] = {}

    def build_prototypes(
        self,
        X_enroll: np.ndarray,
        y_enroll: np.ndarray,
        enrollment_k: Optional[int] = None,
        batch_size: int = 32,
        shuffle_per_user: bool = True,
        random_state: int = 42,
    ) -> Dict[int, np.ndarray]:
        """
        Build one prototype/template per user.

        enrollment_k:
            None -> use all available enrollment samples.
            int  -> use up to k samples per user.
        """

        self.prototypes = {}

        for user_id in self.users:

            indices = np.where(y_enroll == user_id)[0]

            if indices.size == 0:
                continue

            user_samples = X_enroll[indices].copy()

            if shuffle_per_user:
                rng = np.random.default_rng(random_state)
                rng.shuffle(user_samples, axis=0)

            if enrollment_k is not None:
                k = min(enrollment_k, len(user_samples))
                user_samples = user_samples[:k]

            embeddings = self.embedding_extractor.embed(
                user_samples,
                batch_size=batch_size,
            )

            prototype = embeddings.mean(
                axis=0,
                keepdims=True,
            )

            prototype = l2_normalize(
                prototype,
                axis=1,
            )[0]

            self.prototypes[int(user_id)] = prototype

        return self.prototypes

    def get_prototype(
        self,
        user_id: int,
    ) -> Optional[np.ndarray]:
        return self.prototypes.get(int(user_id))


# ============================================================
# Prototype-only verification scorer
# ============================================================

class PrototypeVerificationScorer:
    """
    Creates verification trials using ONLY cosine similarity
    between a sample embedding and the claimed user's prototype.

    No UBM probabilities, log-probabilities, or fusion scores
    are computed anywhere in this class.
    """

    def __init__(
        self,
        embedding_extractor: EmbeddingExtractor,
        prototype_manager: PrototypeManager,
    ):
        self.embedding_extractor = embedding_extractor
        self.prototype_manager = prototype_manager

    def compute_scores(
        self,
        X: np.ndarray,
        y: np.ndarray,
        batch_size: int = 32,
    ) -> Dict[str, np.ndarray]:

        embeddings = self.embedding_extractor.embed(
            X,
            batch_size=batch_size,
        )

        scores = []
        labels = []
        claimed_users = []

        for i in range(len(X)):

            true_user = int(y[i])
            sample_embedding = embeddings[i]

            for user_id in self.prototype_manager.users:

                user_id = int(user_id)
                label = 1 if user_id == true_user else 0

                prototype = self.prototype_manager.get_prototype(
                    user_id
                )

                if prototype is None:
                    continue

                # Cosine similarity because both vectors are L2-normalized.
                cosine_score = float(
                    np.dot(sample_embedding, prototype)
                )

                scores.append(cosine_score)
                labels.append(label)
                claimed_users.append(user_id)

        return {
            "scores": np.asarray(scores, dtype=np.float32),
            "labels": np.asarray(labels, dtype=np.int32),
            "claimed_users": np.asarray(
                claimed_users,
                dtype=np.int32,
            ),
        }

In [ ]:
# ============================================================
# Experiment runner
# ============================================================

class PrototypeExperimentRunner:

    def __init__(
        self,
        classifier_model: tf.keras.Model,
        X_set1: np.ndarray,
        y_set1: np.ndarray,
        X_set2: np.ndarray,
        y_set2: np.ndarray,
        exp_root: str = "auth_experiments",
        exp_name: str = "proto_templates",
        config: Optional[ExperimentConfig] = None,
    ):

        self.classifier_model = classifier_model

        self.X_set1 = X_set1
        self.y_set1 = y_set1

        self.X_set2 = X_set2
        self.y_set2 = y_set2

        self.users = np.unique(
            np.concatenate([y_set1, y_set2])
        ).astype(int)

        self.config = config or ExperimentConfig()

        self.paths = ExperimentPaths(
            exp_root,
            exp_name,
        )

        self._save_json(
            asdict(self.config),
            self.paths.path_config(
                "experiment_config.json"
            ),
        )

        self.embedding_extractor: Optional[
            EmbeddingExtractor
        ] = None

        self.prototype_manager: Optional[
            PrototypeManager
        ] = None

        self.scorer: Optional[
            PrototypeVerificationScorer
        ] = None

        self._make_dev_test_split()

    # --------------------------------------------------------
    # File utilities
    # --------------------------------------------------------

    @staticmethod
    def _save_json(obj, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2)

    @staticmethod
    def _save_pickle(obj, path: str):
        with open(path, "wb") as f:
            pickle.dump(obj, f)

    # --------------------------------------------------------
    # Development / final test split
    # --------------------------------------------------------

    def _make_dev_test_split(self):

        sss = StratifiedShuffleSplit(
            n_splits=1,
            test_size=self.config.dev_test_split,
            random_state=self.config.random_state,
        )

        dev_idx, test_idx = next(
            sss.split(
                self.X_set2,
                self.y_set2,
            )
        )

        self.X_dev = self.X_set2[dev_idx]
        self.y_dev = self.y_set2[dev_idx]

        self.X_test_final = self.X_set2[test_idx]
        self.y_test_final = self.y_set2[test_idx]

        info = {
            "X_set1_shape": self.X_set1.shape,
            "X_set2_shape": self.X_set2.shape,
            "X_dev_shape": self.X_dev.shape,
            "X_test_final_shape": self.X_test_final.shape,
        }

        self._save_json(
            info,
            self.paths.path_config(
                "data_shapes.json"
            ),
        )

    # --------------------------------------------------------
    # Main pipeline
    # --------------------------------------------------------

    def run(self):

        # 1. Create embedding extractor.
        self.embedding_extractor = EmbeddingExtractor(
            self.classifier_model,
            use_pooling=self.config.use_pooling_for_embedding,
        )

        # 2. Create prototype manager.
        self.prototype_manager = PrototypeManager(
            self.embedding_extractor,
            self.users,
        )

        # 3. Create prototype-only scorer.
        self.scorer = PrototypeVerificationScorer(
            self.embedding_extractor,
            self.prototype_manager,
        )

        # 4. Build full enrollment templates.
        self.prototype_manager.build_prototypes(
            self.X_set1,
            self.y_set1,
            enrollment_k=None,
            batch_size=self.config.batch_size,
            shuffle_per_user=True,
            random_state=self.config.random_state,
        )

        self._save_pickle(
            self.prototype_manager.prototypes,
            self.paths.path_data(
                "prototypes_full.pkl"
            ),
        )

        # 5. Development phase:
        #    determine the EER operating threshold.
        (
            dev_results,
            threshold,
            dev_eer,
            roc_info,
        ) = self._dev_phase()

        self._save_dev_outputs(
            dev_results,
            threshold,
            dev_eer,
            roc_info,
        )

        # 6. Enrollment-size study.
        eer_vs_k = self._enrollment_size_study()

        self._save_enroll_size_outputs(
            eer_vs_k
        )

        # 7. Final test evaluation.
        (
            test_results,
            final_metrics_overall,
            final_metrics_per_user,
        ) = self._final_test_phase(
            threshold
        )

        self._save_test_outputs(
            test_results,
            final_metrics_overall,
            final_metrics_per_user,
        )

        print(
            f"\nExperiment completed: {self.paths.exp_dir}"
        )

    # --------------------------------------------------------
    # Development phase
    # --------------------------------------------------------

    def _dev_phase(self):

        assert self.scorer is not None

        dev_results = self.scorer.compute_scores(
            self.X_dev,
            self.y_dev,
            batch_size=self.config.batch_size,
        )

        scores = dev_results["scores"]
        labels = dev_results["labels"]

        (
            threshold,
            eer,
            roc_auc,
            fpr,
            tpr,
            thresholds,
        ) = compute_eer_threshold(
            labels,
            scores,
        )

        roc_info = {
            "fpr": fpr,
            "tpr": tpr,
            "auc": roc_auc,
            "thr": thresholds,
        }

        print(
            f"[DEV] PROTO EER={eer:.4f}, "
            f"thr={threshold:.4f}, "
            f"AUC={roc_auc:.4f}"
        )

        return (
            dev_results,
            threshold,
            eer,
            roc_info,
        )

    # --------------------------------------------------------
    # Save development outputs
    # --------------------------------------------------------

    def _save_dev_outputs(
        self,
        dev_results,
        threshold,
        eer,
        roc_info,
    ):

        df_dev = pd.DataFrame(
            {
                "system": "proto",
                "score": dev_results["scores"],
                "label": dev_results["labels"],
                "claimed_user": dev_results["claimed_users"],
            }
        )

        df_dev.to_csv(
            self.paths.path_data(
                "dev_scores_labels.csv"
            ),
            index=False,
        )

        df_summary = pd.DataFrame(
            [
                {
                    "system": "proto",
                    "EER": eer,
                    "threshold_EER": threshold,
                    "AUC": roc_info["auc"],
                }
            ]
        )

        df_summary.to_csv(
            self.paths.path_data(
                "dev_summary_metrics.csv"
            ),
            index=False,
        )

        threshold_config = {
            "system": "proto",
            "threshold_EER": threshold,
            "EER": eer,
            "AUC": roc_info["auc"],
        }

        self._save_json(
            threshold_config,
            self.paths.path_config(
                "proto_threshold.json"
            ),
        )

        np.save(
            self.paths.path_data(
                "roc_proto_fpr.npy"
            ),
            roc_info["fpr"],
        )

        np.save(
            self.paths.path_data(
                "roc_proto_tpr.npy"
            ),
            roc_info["tpr"],
        )

        np.save(
            self.paths.path_data(
                "roc_proto_thr.npy"
            ),
            roc_info["thr"],
        )

        # ROC plot
        plt.figure()

        plt.plot(
            roc_info["fpr"],
            roc_info["tpr"],
            label=f"Proto (AUC={roc_info['auc']:.3f})",
        )

        plt.plot(
            [0, 1],
            [0, 1],
            "k--",
        )

        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC on Dev Set: Prototype-Based Authentication")
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.tight_layout()

        plt.savefig(
            self.paths.path_plot(
                "roc_dev_proto.png"
            ),
            dpi=300,
        )

        plt.savefig(
            self.paths.path_plot(
                "roc_dev_proto.pdf"
            )
        )

        plt.close()

    # --------------------------------------------------------
    # Enrollment-size study
    # --------------------------------------------------------

    def _enrollment_size_study(self):

        assert self.scorer is not None
        assert self.prototype_manager is not None

        records = []

        for k in self.config.enroll_sizes:

            self.prototype_manager.build_prototypes(
                self.X_set1,
                self.y_set1,
                enrollment_k=k,
                batch_size=self.config.batch_size,
                shuffle_per_user=True,
                random_state=self.config.random_state,
            )

            self._export_templates_after_dev_proto(
                subdir_name=f"kEnrollments/k_{k}"
            )

            proto_scores = self.scorer.compute_scores(
                self.X_dev,
                self.y_dev,
                batch_size=self.config.batch_size,
            )

            scores = proto_scores["scores"]
            labels = proto_scores["labels"]

            (
                threshold,
                eer,
                roc_auc,
                _,
                _,
                _,
            ) = compute_eer_threshold(
                labels,
                scores,
            )

            records.append(
                {
                    "enroll_k": int(k),
                    "EER": float(eer),
                    "threshold_EER": float(threshold),
                    "AUC": float(roc_auc),
                }
            )

            print(
                f"[DEV] Proto-only, "
                f"enroll_k={k} -> "
                f"EER={eer:.4f}, "
                f"thr={threshold:.4f}, "
                f"AUC={roc_auc:.4f}"
            )

        return records

    def _export_templates_after_dev_proto(
        self,
        subdir_name: str,
    ):
        """
        Save enrollment templates for each enrollment-size
        experiment.
        """

        assert self.prototype_manager is not None

        output_dir = os.path.join(
            self.paths.data_dir,
            subdir_name,
        )

        os.makedirs(
            output_dir,
            exist_ok=True,
        )

        self._save_pickle(
            self.prototype_manager.prototypes,
            os.path.join(
                output_dir,
                "prototypes.pkl",
            ),
        )

    def _save_enroll_size_outputs(
        self,
        records,
    ):

        df_eer_vs_k = pd.DataFrame(records)

        df_eer_vs_k.to_csv(
            self.paths.path_data(
                "dev_eer_vs_enrollment_k.csv"
            ),
            index=False,
        )

        np.save(
            self.paths.path_data(
                "dev_enroll_k.npy"
            ),
            df_eer_vs_k["enroll_k"].to_numpy(),
        )

        np.save(
            self.paths.path_data(
                "dev_eer_vs_k.npy"
            ),
            df_eer_vs_k["EER"].to_numpy(),
        )

        plt.figure()

        plt.plot(
            df_eer_vs_k["enroll_k"],
            df_eer_vs_k["EER"],
            marker="o",
        )

        plt.xlabel(
            "Number of Enrollment Samples per User (k)"
        )
        plt.ylabel("EER (Dev Set)")
        plt.title(
            "Prototype System: EER vs Enrollment Size"
        )
        plt.grid(True)
        plt.tight_layout()

        plt.savefig(
            self.paths.path_plot(
                "eer_vs_enrollment_k.png"
            ),
            dpi=300,
        )

        plt.savefig(
            self.paths.path_plot(
                "eer_vs_enrollment_k.pdf"
            )
        )

        plt.close()

        self._save_pickle(
            records,
            self.paths.path_data(
                "dev_eer_vs_enrollment_k.pkl"
            ),
        )

    # --------------------------------------------------------
    # Final test phase
    # --------------------------------------------------------

    def _final_test_phase(
        self,
        threshold: float,
    ):

        assert self.scorer is not None
        assert self.prototype_manager is not None

        # Rebuild templates using all enrollment samples.
        self.prototype_manager.build_prototypes(
            self.X_set1,
            self.y_set1,
            enrollment_k=None,
            batch_size=self.config.batch_size,
            shuffle_per_user=True,
            random_state=self.config.random_state,
        )

        test_results = self.scorer.compute_scores(
            self.X_test_final,
            self.y_test_final,
            batch_size=self.config.batch_size,
        )

        scores = test_results["scores"]
        labels = test_results["labels"]
        claimed = test_results["claimed_users"]

        overall = eval_at_threshold(
            labels,
            scores,
            threshold,
        )

        overall["system"] = "proto"
        overall["threshold"] = threshold

        final_metrics_overall = [overall]
        final_metrics_per_user = []

        for user_id in self.users:

            mask = claimed == int(user_id)

            if mask.sum() == 0:
                continue

            user_labels = labels[mask]
            user_scores = scores[mask]

            user_metrics = eval_at_threshold(
                user_labels,
                user_scores,
                threshold,
            )

            user_metrics["system"] = "proto"
            user_metrics["user"] = int(user_id)
            user_metrics["threshold"] = threshold

            final_metrics_per_user.append(
                user_metrics
            )

        return (
            test_results,
            final_metrics_overall,
            final_metrics_per_user,
        )

    # --------------------------------------------------------
    # Save final test outputs
    # --------------------------------------------------------

    def _save_test_outputs(
        self,
        test_results,
        final_metrics_overall,
        final_metrics_per_user,
    ):

        df_test = pd.DataFrame(
            {
                "system": "proto",
                "score": test_results["scores"],
                "label": test_results["labels"],
                "claimed_user": test_results["claimed_users"],
            }
        )

        df_test.to_csv(
            self.paths.path_data(
                "test_scores_labels.csv"
            ),
            index=False,
        )

        df_test_overall = pd.DataFrame(
            final_metrics_overall
        )

        df_test_overall.to_csv(
            self.paths.path_data(
                "test_overall_metrics.csv"
            ),
            index=False,
        )

        df_test_per_user = pd.DataFrame(
            final_metrics_per_user
        )

        df_test_per_user.to_csv(
            self.paths.path_data(
                "test_per_user_metrics.csv"
            ),
            index=False,
        )

        self._save_pickle(
            {
                "overall": final_metrics_overall,
                "per_user": final_metrics_per_user,
            },
            self.paths.path_data(
                "test_metrics.pkl"
            ),
        )


---

In [ ]:
# ============================================================
# Example run
# ============================================================

DIR_PREFIX = "<path to base experiment directory>"
MODEL_DIR = f"{DIR_PREFIX}/<path to export Bi-GRU model>"
EXP_PATH = f"{DIR_PREFIX}/<location to export results and templates>"

SAMPLE_PATH = f"{DIR_PREFIX}/<path to load KAN-AE encoded data>"

In [ ]:
# Trained classification model.
# It is used ONLY to extract the embedding representation.
MODEL_PATH = f"{MODEL_DIR}/bigru_classifier.keras"

classifier_model = tf.keras.models.load_model(
    MODEL_PATH
)

In [ ]:
# SET1: enrollment data.
X_enroll = np.load(f"{SAMPLE_PATH}/X_val_pca_enc.npy")
y_enroll = np.load(f"{SAMPLE_PATH}/y_val.npy")

# SET2: verification data.
X_test = np.load(f"{SAMPLE_PATH}/X_test_pca_enc.npy")
y_test = np.load(f"{SAMPLE_PATH}/y_test.npy")


print(
    "Enroll/Val:",
    X_enroll.shape,
    y_enroll.shape,
)

print(
    "Test:",
    X_test.shape,
    y_test.shape,
)

In [ ]:
## main run
runner = PrototypeExperimentRunner(
    classifier_model=classifier_model,
    X_set1=X_enroll,
    y_set1=y_enroll,
    X_set2=X_test,
    y_set2=y_test,
    exp_root=f"{EXP_PATH}/auth_experiments",
    exp_name="proto_templates",
)

runner.run()

---